# INT8 Quantization Smoke Test (bitsandbytes)

**Purpose:** Determine whether INT8 quantization of Stable Diffusion v1.5 is feasible on the 2080 Ti via bitsandbytes, with hard time-boxing.

**Out of scope:** Integration into the main benchmark harness, statistical timing, CLIP score evaluation, nsys profiling. This is exploratory-only.

**Time budget:** 30 minutes. If you hit major debugging at 30 min, stop and report "INT8 was attempted but blocked by [specific issue]" in the v1 draft. That's strictly better than the current vague "compatibility issues" sentence.

**Honest expectation:** Likely to partially succeed at best. bitsandbytes' default 8-bit path quantizes `nn.Linear` layers, but SD's UNet is dominated by `nn.Conv2d` layers. Best case: ~1.1-1.4x speedup over FP16 from partial quantization. Worst case: bitsandbytes errors out on SD's UNet and we can't even load the model. Either outcome is reportable.

**Reference baseline:** FP16 SDPA (config 05) achieves ~1.83 s/image. This is what we're comparing against.

## Step 1: Install bitsandbytes

If this fails (common: CUDA version mismatch with 12.4), the smoke test is over. Report that as the result.

In [ ]:
import subprocess
import sys

# Use --no-deps to avoid bitsandbytes pulling in conflicting torch versions
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'bitsandbytes==0.43.3', '--no-deps'],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    print()
    print('Install failed. Smoke test result: NOT FEASIBLE - bitsandbytes installation failed.')
    print('Specific error to report in v1 draft above.')

In [ ]:
# Verify bitsandbytes loads and detects the GPU correctly.
# This is the first place sm_75 (Turing) compatibility issues show up.
try:
    import bitsandbytes as bnb
    print(f'bitsandbytes version: {bnb.__version__}')
    
    # Force a CUDA op to verify the library actually works on this GPU
    import torch
    if torch.cuda.is_available():
        # Try a basic 8-bit linear layer
        layer = bnb.nn.Linear8bitLt(64, 32, has_fp16_weights=False).cuda()
        x = torch.randn(4, 64, dtype=torch.float16, device='cuda')
        y = layer(x)
        print(f'Basic 8-bit linear test PASSED. Output shape: {y.shape}, dtype: {y.dtype}')
    else:
        print('CUDA not available - aborting smoke test')
except Exception as e:
    print(f'bitsandbytes import or basic test FAILED: {type(e).__name__}: {e}')
    print()
    print('Smoke test result: NOT FEASIBLE - bitsandbytes does not work on this GPU/setup.')

## Step 2: Establish FP16 reference baseline

Before testing INT8, generate one FP16 image with the same prompt/seed so we have a head-to-head comparison. This avoids ambiguity about whether INT8 timing differences are due to the model or the test setup.

Hard reset GPU 0 first to make sure we're not measuring leftover state.

In [ ]:
import gc
import torch
import time
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

# Hard reset GPU
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f'GPU memory before FP16 load: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated')

In [ ]:
# Same model + prompt as our existing benchmarks for direct comparison
MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
PROMPT = 'a photograph of an astronaut riding a horse on the moon, highly detailed'
SEED = 42
STEPS = 25

# Build FP16 reference pipeline
pipe_fp16 = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe_fp16.scheduler = DPMSolverMultistepScheduler.from_config(pipe_fp16.scheduler.config)
pipe_fp16 = pipe_fp16.to('cuda:0')
pipe_fp16.set_progress_bar_config(disable=True)

# Use SDPA (best single-GPU config)
from diffusers.models.attention_processor import AttnProcessor2_0
pipe_fp16.unet.set_attn_processor(AttnProcessor2_0())

print(f'GPU memory after FP16 load: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated')

In [ ]:
# Warmup + 3 timed runs for FP16 reference
def time_inference(pipe, prompt, seed, steps, device='cuda:0'):
    gen = torch.Generator(device=device).manual_seed(seed)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    img = pipe(prompt, num_inference_steps=steps, generator=gen).images[0]
    torch.cuda.synchronize()
    return time.perf_counter() - t0, img

print('FP16 warmup...')
_ = time_inference(pipe_fp16, PROMPT, SEED, STEPS)
_ = time_inference(pipe_fp16, PROMPT, SEED, STEPS)

fp16_times = []
for i in range(3):
    t, img_fp16 = time_inference(pipe_fp16, PROMPT, SEED, STEPS)
    fp16_times.append(t)
    print(f'  FP16 SDPA iter {i+1}: {t:.2f} s')

fp16_mean = sum(fp16_times) / len(fp16_times)
print(f'\nFP16 SDPA mean: {fp16_mean:.3f} s/image (3 timed runs)')
img_fp16.save('int8_smoke_fp16_reference.png')
img_fp16

In [ ]:
# Free FP16 pipeline before loading INT8 - 2080 Ti has 11 GB VRAM
del pipe_fp16
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f'GPU memory after free: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated')

## Step 3: Build INT8-quantized pipeline

**This is where most failures happen.** bitsandbytes' `replace_with_bnb_linear` only converts `nn.Linear` modules. SD's UNet is mostly `nn.Conv2d`, so we get partial quantization at best.

Approach: load FP16 weights, then walk the UNet and replace eligible Linear layers with `Linear8bitLt`. Keep VAE and text encoder in FP16 (they're small and quantizing them isn't standard practice for SD).

**If this cell errors out, that's a reportable result.** Capture the exact error message.

In [ ]:
import bitsandbytes as bnb
import torch.nn as nn

# Re-load the pipeline in FP16 first (we'll quantize selectively)
pipe_int8 = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe_int8.scheduler = DPMSolverMultistepScheduler.from_config(pipe_int8.scheduler.config)

# Count Linear vs Conv2d layers in UNet to set expectations
linear_count = sum(1 for m in pipe_int8.unet.modules() if isinstance(m, nn.Linear))
conv_count = sum(1 for m in pipe_int8.unet.modules() if isinstance(m, nn.Conv2d))
total_params = sum(p.numel() for p in pipe_int8.unet.parameters())
linear_params = sum(p.numel() for m in pipe_int8.unet.modules() if isinstance(m, nn.Linear) for p in m.parameters())
conv_params = sum(p.numel() for m in pipe_int8.unet.modules() if isinstance(m, nn.Conv2d) for p in m.parameters())

print(f'UNet structure:')
print(f'  Linear layers:  {linear_count} ({linear_params/1e6:.1f}M params, {linear_params/total_params*100:.1f}%)')
print(f'  Conv2d layers:  {conv_count} ({conv_params/1e6:.1f}M params, {conv_params/total_params*100:.1f}%)')
print(f'  Total params:   {total_params/1e6:.1f}M')
print()
print(f'Predicted INT8 coverage: ~{linear_params/total_params*100:.0f}% of UNet weights.')
print(f'Conv layers stay in FP16 (bitsandbytes does not quantize Conv2d).')
print(f'Speedup will be capped accordingly.')

In [ ]:
# Replace Linear layers with Linear8bitLt in the UNet
# This is the moment where unsupported ops typically blow up

def replace_linear_with_int8(module, parent_name=''):
    """Walk the module tree and replace nn.Linear with bnb.Linear8bitLt in-place.
    Returns count of replacements."""
    n_replaced = 0
    for name, child in list(module.named_children()):
        full_name = f'{parent_name}.{name}' if parent_name else name
        if isinstance(child, nn.Linear):
            new_layer = bnb.nn.Linear8bitLt(
                child.in_features,
                child.out_features,
                bias=child.bias is not None,
                has_fp16_weights=False,  # actual INT8 weights, not FP16 stored as INT8
                threshold=6.0,  # outlier threshold (LLM.int8() default)
            )
            # Copy weights
            new_layer.weight = bnb.nn.Int8Params(
                child.weight.data.clone(),
                requires_grad=False,
                has_fp16_weights=False,
            )
            if child.bias is not None:
                new_layer.bias = nn.Parameter(child.bias.data.clone())
            setattr(module, name, new_layer)
            n_replaced += 1
        else:
            n_replaced += replace_linear_with_int8(child, full_name)
    return n_replaced

try:
    # Move UNet to CPU first - bnb needs to quantize on first .cuda() call
    pipe_int8.unet = pipe_int8.unet.cpu()
    n_replaced = replace_linear_with_int8(pipe_int8.unet)
    print(f'Replaced {n_replaced} Linear layers with Linear8bitLt')
    
    # Now move everything to GPU - this triggers the actual quantization
    pipe_int8 = pipe_int8.to('cuda:0')
    pipe_int8.set_progress_bar_config(disable=True)
    pipe_int8.unet.set_attn_processor(AttnProcessor2_0())
    
    print(f'GPU memory after INT8 quantization: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated')
    print('SUCCESS: INT8 pipeline built without errors')
except Exception as e:
    print(f'INT8 PIPELINE BUILD FAILED: {type(e).__name__}: {e}')
    print()
    print('Smoke test result: NOT FEASIBLE - INT8 quantization of SD 1.5 UNet failed.')
    print('Capture this exact error message for the v1 draft.')

## Step 4: Run INT8 inference and measure

If pipeline build succeeded, generate one image and time it. If output is garbled noise, that's also a reportable failure mode ("runs but produces unusable output").

In [ ]:
try:
    print('INT8 warmup...')
    _ = time_inference(pipe_int8, PROMPT, SEED, STEPS)
    _ = time_inference(pipe_int8, PROMPT, SEED, STEPS)
    
    int8_times = []
    for i in range(3):
        t, img_int8 = time_inference(pipe_int8, PROMPT, SEED, STEPS)
        int8_times.append(t)
        print(f'  INT8 SDPA iter {i+1}: {t:.2f} s')
    
    int8_mean = sum(int8_times) / len(int8_times)
    print(f'\nINT8 SDPA mean: {int8_mean:.3f} s/image (3 timed runs)')
    
    img_int8.save('int8_smoke_int8_output.png')
except Exception as e:
    print(f'INT8 INFERENCE FAILED: {type(e).__name__}: {e}')
    print()
    print('Smoke test result: PARTIALLY FEASIBLE - pipeline builds but inference errors.')
    img_int8 = None
    int8_mean = None

In [ ]:
# Display INT8 output if it exists
img_int8

## Step 5: Side-by-side comparison

Visual check: did INT8 produce a reasonable image, or noise/artifacts? This determines whether INT8 is reportable as a working configuration or as a failed one.

In [ ]:
import matplotlib.pyplot as plt

if img_int8 is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(img_fp16)
    axes[0].set_title(f'FP16 SDPA: {fp16_mean:.2f} s/image')
    axes[0].axis('off')
    axes[1].imshow(img_int8)
    axes[1].set_title(f'INT8 mixed: {int8_mean:.2f} s/image')
    axes[1].axis('off')
    plt.tight_layout()
    plt.savefig('int8_smoke_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()
    
    print(f'\n=== SMOKE TEST RESULTS ===')
    print(f'FP16 SDPA mean:    {fp16_mean:.3f} s/image')
    print(f'INT8 mixed mean:   {int8_mean:.3f} s/image')
    print(f'INT8 vs FP16:      {fp16_mean/int8_mean:.3f}x speedup' if int8_mean < fp16_mean else f'INT8 vs FP16:      {fp16_mean/int8_mean:.3f}x (slower)')
    print(f'\nVisual quality check: examine images above for artifacts.')
else:
    print('Skipping side-by-side - INT8 inference did not complete.')
    print(f'\n=== SMOKE TEST RESULTS ===')
    print(f'FP16 SDPA mean: {fp16_mean:.3f} s/image (reference only)')
    print(f'INT8: did not produce output')

## Step 6: Summary for the report

Whatever the outcome, this notebook produces concrete evidence to cite. Three possible reportable outcomes:

**Outcome A: INT8 ran, produced reasonable image, was faster than FP16.**
Report: "We performed an exploratory INT8 quantization test using bitsandbytes 0.43.3, replacing nn.Linear layers in the UNet with Linear8bitLt while keeping convolutions in FP16. This achieved Xs/image (Yx vs FP16 SDPA), with no perceptible quality regression in the test prompt. Full benchmarking is left as future work because bitsandbytes only quantizes Linear layers (~Z% of UNet parameters), and a full INT8 path through Conv2d layers would require alternative tools such as TensorRT."

**Outcome B: INT8 ran but produced garbled output, OR was slower than FP16.**
Report: "We attempted INT8 quantization via bitsandbytes 0.43.3. The pipeline executed but produced [degraded outputs / longer latency than FP16] due to [the partial coverage of bitsandbytes for Linear-only layers / outlier handling overhead exceeding INT8 compute savings on this workload]. This confirms the architectural limitation discussed in Section 4 and motivates future work on diffusion-specific quantization frameworks."

**Outcome C: INT8 failed to load.**
Report: "INT8 quantization was attempted via bitsandbytes 0.43.3 but failed at [specific stage: install / model load / inference], with error: [exact error]. This is consistent with the library's transformer-focused design and confirms the integration complexity referenced in Section 2.1."

Pick whichever matches your result and add it to the v1 draft. Don't overstate. Don't extrapolate to claims you don't have data for.

In [ ]:
# Cleanup
if 'pipe_int8' in dir():
    del pipe_int8
gc.collect()
torch.cuda.empty_cache()
print('Cleanup done. Note: bitsandbytes is now installed in the sd-profiling env. Removing it requires:')
print('  pip uninstall bitsandbytes -y')